In [1]:
:set -XRankNTypes
:set -XTypeOperators
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XInstanceSigs
:set -XScopedTypeVariables
:set -XMultiParamTypeClasses
:set -XDeriveFunctor
import Data.Char (toUpper, toLower)
import Data.List (intercalate)
putStrLn "Setup done." 

Setup done.

# ↔️ Профункторы в Haskell

**Профункторы** — недостающее звено между функторами и оптиками.

Профунктор `p :: * -> * -> *` — это конструктор типов,
контравариантный по первому аргументу и ковариантный по второму:

```haskell
class Profunctor p where
  dimap :: (a' -> a) -> (b -> b') -> p a b -> p a' b'
  lmap  :: (a' -> a) -> p a b -> p a' b
  rmap  :: (b -> b') -> p a b -> p a b'
```

> **📦 Зависимости**
> **Пакеты:** `base`
> **Расширения:** `DeriveFunctor` — deriving Functor ([→](Extensions.ipynb#deriving)) · `ExistentialQuantification` — скрытие типа за конструктором (forall в data) ([→](Extensions.ipynb#existentialquantification)) · `FlexibleContexts` — произвольные ограничения в контекстах ([→](Extensions.ipynb#flexiblecontexts)) · `FlexibleInstances` — инстансы для конкретных типов-применений ([→](Extensions.ipynb#flexibleinstances)) · `InstanceSigs` — сигнатуры методов прямо в instance ([→](Extensions.ipynb#instancesigs)) · `MultiParamTypeClasses` — классы с несколькими параметрами ([→](Extensions.ipynb#multiparamtypeclasses)) · `RankNTypes` — forall внутри аргументов функций ([→](Extensions.ipynb#rankntypes)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables)) · `TypeOperators` — операторы на уровне типов: f :+: g ([→](Extensions.ipynb#typeoperators))


## 📌 Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | Профунктор: `dimap` | контравариантность слева, ковариантность справа |
| 2 | Категориальный взгляд | функтор из `Hask^op × Hask` |
| 3 | Подклассы | Strong, Choice, Closed, Costrong, Cochoice |
| 4 | Оптики через профункторы | `forall p. C p => p a b -> p s t` |
| 5 | Призма | Choice-оптика для типов-сумм |
| 6 | Библиотека `Data.Profunctor` | Star, Costar, Forget, Kleisli |
| 7 | Бикатегория Prof | профункторы как морфизмы |
| 8 | Ends и Coends | `forall x. p x x`, wedge-условие |
| 9 | Модули Тамбары | почему профункторные оптики работают формально |

## 1️⃣ Профунктор: `dimap`, контравариантность и ковариантность

### Определение

```
class Profunctor p where
  dimap :: (a' -> a) -> (b -> b') -> p a b -> p a' b'
```

Мнемоника: `dimap f g = lmap f . rmap g`

### Простейший пример: функции `(->)`

```haskell
instance Profunctor (->) where
  dimap f g h = g . h . f
  -- предобработка входа (f), постобработка выхода (g)
```

In [2]:
:set -XRankNTypes
-- Define Profunctor class
class Profunctor p where
  dimap :: (a' -> a) -> (b -> b') -> p a b -> p a' b'
  lmap  :: (a' -> a) -> p a b -> p a' b
  lmap f = dimap f id
  rmap  :: (b -> b') -> p a b -> p a b'
  rmap g = dimap id g

-- The canonical profunctor: (->)
instance Profunctor (->) where
  dimap f g h = g . h . f
  lmap  f h   = h . f
  rmap  g h   = g . h

-- Test
double :: Int -> Int
double x = x * 2

addBang :: String -> String
addBang s = s ++ "!"

-- dimap: pre-map length (String->Int), double it (Int->Int), show (Int->String)
-- starting from (id :: Int->Int), pre-compose with length, post-compose with show.double
f1 :: String -> String
f1 = dimap length (show . double) (id :: Int -> Int)

-- lmap: pre-compose negate before show
f2 :: Int -> String
f2 = lmap negate (show :: Int -> String)

-- rmap: post-compose addBang after (show . length)
f3 :: String -> String
f3 = rmap addBang (show . length)

let w1 = f1 "hello"
let w2 = f2 42
let w3 = f3 "hello"
putStrLn $ "dimap length (show.double) id $ hello = " ++ w1
putStrLn $ "lmap negate show $ 42 = " ++ w2
putStrLn $ "rmap addBang (show.length) $ hello = " ++ w3

dimap length (show.double) id $ hello = 10

lmap negate show $ 42 = -42

rmap addBang (show.length) $ hello = 5!

## 🔴Категориальный взгляд: `Hask^op x Hask -> Hask`

Профунктор `p` — это функтор из **произведения категорий** `Hask^op x Hask` в `Hask`:

![Profunctor dimap kvadrat](../diagrams/optics/prof_dimap.svg)

**Законы профунктора:**
- `dimap id id = id`
- `dimap (f . g) (h . i) = dimap g h . dimap f i`

### Диаграмма: `dimap` как квадрат

![Profunctor dimap квадрат: контравариантно по a, ковариантно по b](../diagrams/optics/dimap_square.svg)

`dimap f g p = g . p . f`  
`p :: Profunctor p => p a b` живёт в **Haskᵒᵖ × Hask** — категории пар (контравариантный аргумент, ковариантный аргумент).

## 3️⃣ Подклассы профунктора

### Иерархия

```
Profunctor
    |-- Strong       (first', second')
    |-- Choice       (left', right')
    |-- Closed       (closed)
    |-- Costrong     (unfirst, unsecond)
    |-- Cochoice     (unleft, unright)
```

### Strong (с произведениями)

```haskell
class Profunctor p => Strong p where
  first'  :: p a b -> p (a, c) (b, c)
  second' :: p a b -> p (c, a) (c, b)
```

### Choice (с суммами)

```haskell
class Profunctor p => Choice p where
  left'  :: p a b -> p (Either a c) (Either b c)
  right' :: p a b -> p (Either c a) (Either c b)
```

In [5]:
-- Strong instance for (->)
class Profunctor p => Strong p where
  first'  :: p a b -> p (a, c) (b, c)
  second' :: p a b -> p (c, a) (c, b)

instance Strong (->) where
  first'  h (a, c) = (h a, c)
  second' h (c, a) = (c, h a)

-- Choice instance for (->)
class Profunctor p => Choice p where
  left'  :: p a b -> p (Either a c) (Either b c)
  right' :: p a b -> p (Either c a) (Either c b)

instance Choice (->) where
  left' h (Left a)  = Left (h a)
  left' _ (Right c) = Right c
  right' _ (Left c) = Left c
  right' h (Right a) = Right (h a)

-- Closed instance for (->)
class Profunctor p => Closed p where
  closed :: p a b -> p (x -> a) (x -> b)

instance Closed (->) where
  closed h f = h . f

-- Tests
double :: Int -> Int
double x = x * 2

putStrLn "--- Strong ---"
let r1 = first' double (3 :: Int, "hello" :: String)
putStrLn $ "first' double (3, hello) = " ++ show r1
let r2 = second' double ("hi" :: String, 5 :: Int)
putStrLn $ "second' double (hi, 5) = " ++ show r2

putStrLn "--- Choice ---"
let r3 = left' double (Left 3 :: Either Int String)
putStrLn $ "left' double (Left 3) = " ++ show r3
let r4 = left' double (Right "x" :: Either Int String)
putStrLn $ "left' double (Right x) = " ++ show r4

putStrLn "--- Closed ---"
let applyToFive = closed double
let r5 = applyToFive (const (4 :: Int)) (0 :: Int)
putStrLn $ "closed double (const 4) 0 = " ++ show r5

--- Strong ---

first' double (3, hello) = (6,"hello")

second' double (hi, 5) = ("hi",10)

--- Choice ---

left' double (Left 3) = Left 6

left' double (Right x) = Right "x"

--- Closed ---

closed double (const 4) 0 = 8

## 4️⃣ Оптики через профункторы

Ключевая идея: **каждая оптика — это естественное преобразование между профункторами**.

### Кодирование через профунктор

```haskell
type Optic p s t a b = p a b -> p s t
```

Конкретные оптики через подклассы:
- `Lens s t a b     = forall p. Strong p => p a b -> p s t`
- `Prism s t a b    = forall p. Choice p => p a b -> p s t`
- `Traversal s t a b = forall p. (Strong p, Choice p) => p a b -> p s t`

In [6]:
:set -XRankNTypes

-- Optic type synonym (local)
type MyOptic p s t a b = p a b -> p s t

-- Lens = Strong profunctor optic
type MyLens s t a b = forall p. Strong p => MyOptic p s t a b
type MyLens' s a = MyLens s s a a

-- Build a lens from getter and setter
-- Applied to (f :: a->a) and (s :: s), gives: set s (f (get s)) :: t
-- This is exactly 'over': modify the focused value
mkLens :: (s -> a) -> (s -> b -> t) -> MyLens s t a b
mkLens get set pab =
  dimap (\s -> (get s, s))
        (\(b, s) -> set s b)
        (first' pab)

-- Example lenses
lensFst :: MyLens' (a, b) a
lensFst = mkLens fst (\(_, b2) a2 -> (a2, b2))

lensSnd :: MyLens' (a, b) b
lensSnd = mkLens snd (\(a2, _) b2 -> (a2, b2))

-- over: apply f to the focused value
-- For (->), l :: (a->a) -> (s->s), so: overL l f s = l f s
overL :: MyLens' s a -> (a -> a) -> s -> s
overL l f s = l f s

-- set: constant replacement
setL :: MyLens' s a -> a -> s -> s
setL l new = overL l (const new)

-- Note: 'view' (get the value) requires Forget profunctor (see cell 12)

putStrLn "--- Lens: overL and setL ---"
let p = (10 :: Int, 20 :: Int)
putStrLn $ "overL lensFst (+5) (10,20) = " ++ show (overL lensFst (+5) p)
putStrLn $ "overL lensSnd (*3) (10,20) = " ++ show (overL lensSnd (*3) p)
putStrLn $ "setL  lensFst 99   (10,20) = " ++ show (setL  lensFst 99   p)
putStrLn $ "setL  lensSnd 0    (10,20) = " ++ show (setL  lensSnd 0    p)

--- Lens: overL and setL ---

overL lensFst (+5) (10,20) = (15,20)

overL lensSnd (*3) (10,20) = (10,60)

setL  lensFst 99   (10,20) = (99,20)

setL  lensSnd 0    (10,20) = (10,0)

## 5️⃣ Призма: оптика для типов-сумм

`Prism` относится к `Either` так же, как `Lens` к `(,)`.

```haskell
type Prism s t a b = forall p. Choice p => p a b -> p s t

_Left  :: Prism (Either a c) (Either b c) a b
_Right :: Prism (Either c a) (Either c b) a b

preview :: Prism s t a b -> s -> Maybe a
review  :: Prism s t a b -> b -> t
```

In [7]:
:set -XRankNTypes

-- Prism = Choice profunctor optic
type MyPrism s t a b = forall p. Choice p => p a b -> p s t
type MyPrism' s a = MyPrism s s a a

-- Build a prism from review + match
mkPrism :: (b -> t) -> (s -> Either t a) -> MyPrism s t a b
mkPrism build match pab =
  dimap match (either id build) (right' pab)

-- _Just prism
prismJust :: MyPrism' (Maybe a) a
prismJust = mkPrism Just (maybe (Left Nothing) Right)

-- overPrism: use the prism with (->) to modify the focused value
overPrism :: MyPrism' s a -> (a -> a) -> s -> s
overPrism l f s = l f s

putStrLn "--- Prism (over) ---"
let r1 = overPrism prismJust (*2) (Just (5 :: Int))
let r2 = overPrism prismJust (*2) (Nothing :: Maybe Int)
putStrLn $ "overPrism prismJust (*2) (Just 5) = " ++ show r1
putStrLn $ "overPrism prismJust (*2) Nothing  = " ++ show r2

-- reviewPrism: build a value using the prism (works via Tagged in cell 12)
-- For now demonstrate with the build function directly:
putStrLn "--- Prism (build/review) ---"
let r3 = Just (42 :: Int)   -- review prismJust 42
putStrLn $ "review prismJust 42 = " ++ show r3
let r4 = Nothing :: Maybe Int  -- no match
putStrLn $ "no match case = " ++ show r4

--- Prism (over) ---

overPrism prismJust (*2) (Just 5) = Just 10

overPrism prismJust (*2) Nothing  = Nothing

--- Prism (build/review) ---

review prismJust 42 = Just 42

no match case = Nothing

## 6️⃣ Библиотека `Data.Profunctor`

Пакет `profunctors` предоставляет полную иерархию.

### Ключевые типы

```haskell
-- Обёртка Kleisli для монады
newtype Kleisli m a b = Kleisli { runKleisli :: a -> m b }

-- Star / Costar
newtype Star f a b     = Star { runStar :: a -> f b }
newtype Costar f a b   = Costar { runCostar :: f a -> b }

-- Потеря информации
data Forget r a b = Forget { runForget :: a -> r }
```

In [8]:
:set -XRankNTypes

-- Forget profunctor: p a b = a -> r (ignores b)
newtype Forget r a b = Forget { runForget :: a -> r }

instance Profunctor (Forget r) where
  dimap f _ (Forget k) = Forget (k . f)

instance Strong (Forget r) where
  first'  (Forget k) = Forget (k . fst)
  second' (Forget k) = Forget (k . snd)

-- view via Forget: a Lens' s a applied to (Forget id) gives (Forget getter)
viewF :: (forall p. Strong p => p a a -> p s s) -> s -> a
viewF l s = runForget (l (Forget id)) s

-- Tagged profunctor: p a b = b (ignores a)
newtype Tagged a b = Tagged { unTagged :: b }

instance Profunctor Tagged where
  dimap _ g (Tagged b) = Tagged (g b)

instance Choice Tagged where
  left'  (Tagged b) = Tagged (Left b)
  right' (Tagged b) = Tagged (Right b)

-- review via Tagged: a Prism' s a applied to (Tagged a) gives (Tagged s)
reviewT :: (forall p. Choice p => p a a -> p s s) -> a -> s
reviewT l a = unTagged (l (Tagged a))

-- Lenses from cell 8 (redefined here):
lensFst2 :: forall p a b. Strong p => p a a -> p (a,b) (a,b)
lensFst2 pab = dimap (\(a,b) -> (a,b)) id (first' pab)

lensSnd2 :: forall p a b. Strong p => p b b -> p (a,b) (a,b)
lensSnd2 pab = dimap (\(a,b) -> (b,a)) (\(b,a) -> (a,b)) (first' pab)

-- Prism from cell 10 (redefined):
prismJust2 :: forall p a. Choice p => p a a -> p (Maybe a) (Maybe a)
prismJust2 = mkPrism Just (maybe (Left Nothing) Right)

-- Test Forget
pair :: (Int, String)
pair = (42, "hello")
putStrLn "--- Forget (view) ---"
putStrLn $ "viewF lensFst2 (42, hello) = " ++ show (viewF lensFst2 pair)
putStrLn $ "viewF lensSnd2 (42, hello) = " ++ show (viewF lensSnd2 pair)

-- Test Tagged (review for prisms)
putStrLn "--- Tagged (review) ---"
let r = reviewT prismJust2 (42 :: Int)
putStrLn $ "reviewT prismJust2 42 = " ++ show r

--- Forget (view) ---

viewF lensFst2 (42, hello) = 42

viewF lensSnd2 (42, hello) = "hello"

--- Tagged (review) ---

reviewT prismJust2 42 = Just 42

In [9]:
-- Star: wraps Kleisli arrows  a -> f b
newtype Star f a b = Star { runStar :: a -> f b }

instance Functor f => Profunctor (Star f) where
  dimap f g (Star h) = Star (fmap g . h . f)

instance Functor f => Strong (Star f) where
  first' (Star h) = Star (\(a, c) -> fmap (\b -> (b, c)) (h a))

-- traverse via Star:
-- traverseOf :: Traversal -> (a -> f b) -> s -> f t
-- traverseOf l f = runStar (l (Star f))

-- Demo: safeHead as a Star Maybe
safeHead :: Star Maybe [a] a
safeHead = Star listToMaybe
  where
    listToMaybe []    = Nothing
    listToMaybe (x:_) = Just x

putStrLn "--- Star (Kleisli arrow as profunctor) ---"
let r1 = runStar safeHead [1::Int,2,3]
let r2 = runStar safeHead ([] :: [Int])
putStrLn $ "runStar safeHead [1,2,3] = " ++ show r1
putStrLn $ "runStar safeHead []      = " ++ show r2

-- dimap over Star: pre-process input, post-process output
-- rmap (*2) safeHead :: Star Maybe [Int] Int
let doubleHead = rmap (*2) safeHead
putStrLn $ "rmap (*2) safeHead [5,6,7] = " ++ show (runStar doubleHead [5::Int,6,7])

-- lmap: apply to a different container type
-- words :: String -> [String], so lmap words safeHead :: Star Maybe String String
let safeFirstWord = lmap words safeHead
putStrLn $ "lmap words safeHead hello-world = " ++ show (runStar safeFirstWord "hello world")
putStrLn $ "lmap words safeHead (empty) = " ++ show (runStar safeFirstWord "")

--- Star (Kleisli arrow as profunctor) ---

runStar safeHead [1,2,3] = Just 1

runStar safeHead []      = Nothing

rmap (*2) safeHead [5,6,7] = Just 10

lmap words safeHead hello-world = Just "hello"

lmap words safeHead (empty) = Nothing

## 7️⃣ Профункторы как морфизмы в бикатегории

### Бикатегория Prof

- **Объекты**: категории (в нашем случае — Hask)
- **Морфизмы**: профункторы `p : C -/-> D`
- **2-морфизмы**: естественные преобразования профункторов

Композиция профункторов определяется через **конец** (end):

```
(p x q)(a, c) = integral^b p(a, b) x q(b, c)
```

Тождественный морфизм — `Hom`-профунктор `Hom(a, b) = a -> b`.

---

## 8. Ends and Coends

### End: the universal wedge

An **end** of a profunctor `p` is a value that simultaneously inhabits `p x x` for *all* `x`,
consistently with all morphisms. In Haskell, this is just `forall`:

```haskell
newtype End p = End { getEnd :: forall x. p x x }
```

**Wedge condition** (naturality in `x`):
for any `f :: a -> b`, the two paths must agree:
```
dimap f id . pi_b  =  dimap id f . pi_a
```
This is automatically satisfied by parametricity in Haskell -- `forall` *is* the end.

### Coend: the universal cowedge

A **coend** is the dual -- an existential:

```haskell
data Coend p = forall x. Coend (p x x)   -- needs ExistentialQuantification
```

### Profunctor composition via coend

The bicategory **Prof** has profunctors as morphisms, composed via coend:

```
(p o q)(a, c)  =  Coend_b (p a b, q b c)
               =  exists b. (p a b, q b c)
```

The identity profunctor is `Hom(a, b) = a -> b`.

### Categorical view

End = limit along the diagonal of the bifunctor `p :: C^op x C -> Set`.
Coend = colimit. In Haskell both are representable via rank-2 types.

![End/Coend wedge diagram](../diagrams/optics/prof_end_wedge.svg)

In [ ]:
:set -XRankNTypes
:set -XExistentialQuantification

-- End: profunctor p, the "diagonal" forall
-- In Haskell, forall x. p x x  IS the end

-- Example: End of Hom = natural transformations
-- End (Hom) = forall x. (x -> x) = identity type

-- Concrete example: End of the function profunctor
newtype EndHom = EndHom { runEndHom :: forall x. x -> x }

idEnd :: EndHom
idEnd = EndHom id

-- The only inhabitant is id (by parametricity)
testEnd :: Int
testEnd = runEndHom idEnd 42

-- Coend example
data SomeState = forall s. SomeState s (s -> String)

-- Coend of p a b = exists x. (a -> x, x -> b)
-- This is the Day convolution structure
data Coend p = forall x. MkCoend (p x x)

-- Profunctor composition: exists b. (p a b, q b c)
data ProfComp p q a c = forall b. ProfComp (p a b) (q b c)

-- Example: compose Maybe-Kleisli with list-Kleisli
step1 :: ProfComp ((->) ) ((->) ) Int String
step1 = ProfComp (show :: Int -> String) (++ "!") 

print testEnd  -- 42
putStrLn "Ends and Coends: OK"

---

## 9. Tambara Modules: why profunctor optics work

### The key question

Why does `forall p. Strong p => p a b -> p s t` *actually encode* a Lens?
The answer: **Tambara modules**.

### Definition

A **Tambara module** for a monoidal action `F` is a profunctor `p` with:

```haskell
class Profunctor p => Tambara p where
  alpha :: p a b -> p (F a) (F b)   -- structural morphism
```

subject to coherence conditions (naturality + unit + associativity).

`Strong` is exactly `Tambara` for the action `F a = (a, c)`:

```haskell
alpha = first :: p a b -> p (a,c) (b,c)
```

`Choice` is `Tambara` for `F a = Either a c`:

```haskell
alpha = left :: p a b -> p (Either a c) (Either b c)
```

### The formal result

By the **Yoneda lemma for profunctors**:

```
(forall p. Tambara_F p => p a b -> p s t)   ~=   Optic_F s t a b
```

A polymorphic function over all Tambara modules for `F` *is exactly* the optic.
This is why the `forall p. Strong p =>` encoding works -- it's not a trick, it's a theorem.

### Categorical view

The category of Tambara modules for `F` is a full subcategory of **Prof**.
An optic = a natural transformation between Tambara modules.
The Yoneda embedding for this subcategory gives the profunctor representation.

![Tambara modules: structural morphism and optic table](../diagrams/optics/prof_tambara.svg)

In [ ]:
:set -XRankNTypes

-- Tambara module = profunctor with structural morphism
-- Strong is Tambara for the (,c) action

class Profunctor p => MyStrong p where
  myFirst :: p a b -> p (a,c) (b,c)

-- (->) is a Strong profunctor
instance MyStrong (->) where
  myFirst f (a, c) = (f a, c)

-- Verify naturality (Tambara condition):
-- myFirst . dimap f g  =  dimap (bimap f id) (bimap g id) . myFirst
testNat :: Bool
testNat =
  let f = (+1) :: Int -> Int
      g = show :: Int -> String
      p = (*2) :: Int -> Int
      lhs = myFirst (dimap f g p) :: (Int, Char) -> (String, Char)
      rhs x = let (a, c) = x
                  (b, c2) = myFirst p (f a, c)
              in (g b, c2)
  in lhs (3, 'x') == rhs (3, 'x')

-- Profunctor optic = forall over Tambara modules
type Lens' s a = forall p. MyStrong p => p a a -> p s s

-- _fst focuses on first element of a pair
_fst :: Lens' (a, b) a
_fst paa = myFirst paa

-- Use it: extract
viewFst :: (a, b) -> a
viewFst = _fst id

-- Use it: modify
overFst :: (a -> a) -> (a,b) -> (a,b)
overFst f = _fst f

print testNat                   -- True
print (viewFst (42, "hello"))   -- 42
print (overFst (*3) (5, True))  -- (15,True)

## Summary

| Concept | Definition |
|---------|------------|
| Profunctor | `dimap :: (a' -> a) -> (b -> b') -> p a b -> p a' b'` |
| Strong | Products: `first :: p a b -> p (a,c) (b,c)` |
| Choice | Sums: `left :: p a b -> p (Either a c) (Either b c)` |
| Lens | `forall p. Strong p => p a b -> p s t` |
| Prism | `forall p. Choice p => p a b -> p s t` |
| End | `forall x. p x x` -- limit along the diagonal |
| Coend | `exists x. p x x` -- colimit, used for profunctor composition |
| Tambara module | profunctor + structural morphism `alpha :: p a b -> p (F a) (F b)` |
| Profunctor optic | `forall p. Tambara_F p => p a b -> p s t` -- Yoneda theorem |

Profunctors -- the powerful abstraction unifying functors, arrows and optics.
The `forall` encoding of optics is not a trick but a consequence of the Yoneda lemma.

## Источники и якоря конструкций

> **★ = наш акцент**; без звёздочки — классика. Категорная нить теории Пытьева — [PytevIso.ipynb](PytevIso.ipynb).

| Конструкция | Якорь |
|---|---|
| Профункторы $C^{op}\times D \to \mathbf{Set}$, `dimap` | Bénabou (distributors); Loregian, *(Co)end Calculus*; nLab «profunctor» |
| Ends/coends | Mac Lane, *CWM*, гл. IX; Loregian |
| Профункторные оптики, модули Тамбары | Pastro–Street (Tambara modules, 2008); Milewski, «Profunctor optics: the categorical view»; Riley, «Categories of Optics» (2018) |
| Бикатегория профункторов $\mathbf{Prof}$ | Bénabou; nLab «Prof» |
| Библиотека `Data.Profunctor` | Kmett, пакет `profunctors` |

---

**← Предыдущий:** [Алгебры и Коалгебры](AlgebrasCoalgebras.ipynb)  |  **Следующий →** [Оптики](Optics.ipynb)
